# Weekly Excel -> Normalized Dataset
This notebook documents a reproducible workflow to load the weekly `sample.xlsx`,
split each sheet into labeled parts using `iloc` ranges, clean the chunks,
concatenate them with day/part labels, and export a normalized workbook.

**Usage**:
1. Ensure `data/sample.xlsx` exists.
2. Activate the project virtualenv and install requirements (see `README.md`).
3. Run cells in order: Loading -> Split -> Clean -> Merge -> Export.

In [36]:
# Environment / imports
# - Requires: pandas, openpyxl, numpy (install via requirements.txt)
import numpy as np
import pandas as pd
import openpyxl as op

# Show pandas version for reproducibility
print('pandas', pd.__version__)


pandas 3.0.1


In [ ]:
# Load the workbook and capture sheet-level metadata (top-left cell) before parsing the table
wb_path = '../data/sample.xlsx'
# Collect sheet names and top-left metadata (SheetDate)
sheets_name = pd.ExcelFile(wb_path, engine='openpyxl').sheet_names
sheets_date = {}
for sheet in ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']:
    if sheet in sheets_name:
        # read first row with no header to capture metadata cell at (0,0)
        meta = pd.read_excel(wb_path, sheet_name=sheet, header=None, nrows=1, engine='openpyxl')
        raw_date = meta.iloc[0,0] if not meta.empty else None
        sheets_date[sheet.lower()] = raw_date
        # now read actual table with header row (adjust header index if necessary)
        df = pd.read_excel(wb_path, sheet_name=sheet, header=3, engine='openpyxl')
        # add a SheetDate column parsed to datetime when possible
        df['SheetDate'] = pd.to_datetime(raw_date, errors='coerce')
        globals()[f'df_{sheet.lower()}'] = df  # creates df_mon, df_tue, etc.
        print(f"Loaded {sheet}: rows={len(df)} cols={len(df.columns)}  SheetDate={raw_date}")

Loaded MON: rows=92 cols=44
Loaded TUE: rows=92 cols=44
Loaded WED: rows=92 cols=44
Loaded THU: rows=92 cols=44
Loaded FRI: rows=92 cols=44
Loaded SAT: rows=92 cols=44
Loaded SUN: rows=92 cols=44


## 2) Split sheets by `iloc` ranges
Define the `iloc` slices you want to extract from each sheet. The code below creates variables
like `df_mon_part1`, `df_mon_part2`, etc., one per slice. Adjust `iloc_ranges` to match your layout.

In [38]:
# Parameters: list of dataframe names / days and iloc ranges
days = ['mon','tue','wed','thu','fri','sat','sun']
# Simple single range example (rows 0:30, cols 0:14)
iloc_ranges = [(0, 30, 0, 15)]
# Or multiple ranges: [(rstart,rend,cstart,cend), ...]
# iloc_ranges = [(0,30,0,14), (30,60,0,14)]

for day in days:
    name = f"df_{day}"
    if name not in globals():
        continue
    df_local = globals()[name]
    for idx, (r0, r1, c0, c1) in enumerate(iloc_ranges, start=1):
        part = df_local.iloc[r0:r1, c0:c1].reset_index(drop=True)
        globals()[f"{name}_part{idx}"] = part
        print(f"{name}_part{idx}: rows {len(part)} cols {len(part.columns)}")

df_mon_part1: rows 30 cols 15
df_tue_part1: rows 30 cols 15
df_wed_part1: rows 30 cols 15
df_thu_part1: rows 30 cols 15
df_fri_part1: rows 30 cols 15
df_sat_part1: rows 30 cols 15
df_sun_part1: rows 30 cols 15


In [39]:
df_mon_part1


,No,Docket No,Reg No,Mobile,Gender,TIME,Service,CASH,CARD,RSL,GV,GV CASH,GV CARD,O/Taker,Comment
0,1,14125,KEN838,0412038383,M,905,PCL,NaN,NaN,0,79,NaN,NaN,HENDRA,22012 (6/6)
1,2,14126,ENA66S,0416951959,F,915,PCL,NaN,NaN,0,79,NaN,475,HENDRA,23225 (1/6)
2,3,14127,EEY61G,0401995714,F,930,X,95,NaN,0,0,NaN,NaN,HENDRA,NaN
3,4,14128,CN800,0418882000,F,935,D,NaN,75,0,0,NaN,NaN,HENDRA,NaN
4,5,14129,JH2211,0412771876,F,945,X,NaN,95,0,0,NaN,NaN,HENDRA,NaN
5,6,14130,CE32DD,0416155783,F,1010,D,NaN,75,0,0,NaN,NaN,SHIVAM,NaN
6,7,14131,EVU18U,0499080777,F,1030,GV,NaN,NaN,0,0,300,NaN,SHIVAM,XD-50-176 TO 181
7,8,14132,EXD61Y,0407213067,F,1050,D,NaN,75,0,0,NaN,NaN,SHIVAM,NaN
8,9,14133,DTY90V,0439592230,M,1100,H,NaN,100,0,0,NaN,NaN,SHIVAM,NaN
9,10,14134,NXQ74D,0408518552,F,1100,X,75,NaN,0,0,NaN,NaN,SHIVAM,NaN


In [40]:
df_mon_part1 = df_mon_part1[df_mon_part1['Docket No'] != 'Docket No'].reset_index(drop=True)
# remove naan from df_mon_part1 docket no
df_mon_part1 = df_mon_part1[df_mon_part1['Docket No'].notna()].reset_index(drop=True)

# automate this for all df_{day}_part{idx} dataframes
for day in days:
    for idx in range(1, len(iloc_ranges)+1):
        name = f"df_{day}_part{idx}"
        if name not in globals():
            continue
        df_local = globals()[name]
        df_local = df_local[df_local['Docket No'] != 'Docket No'].reset_index(drop=True)
        df_local = df_local[df_local['Docket No'].notna()].reset_index(drop=True)
        globals()[name] = df_local

In [41]:
df_mon_part1

,No,Docket No,Reg No,Mobile,Gender,TIME,Service,CASH,CARD,RSL,GV,GV CASH,GV CARD,O/Taker,Comment
0,1,14125,KEN838,0412038383,M,905,PCL,NaN,NaN,0,79,NaN,NaN,HENDRA,22012 (6/6)
1,2,14126,ENA66S,0416951959,F,915,PCL,NaN,NaN,0,79,NaN,475,HENDRA,23225 (1/6)
2,3,14127,EEY61G,0401995714,F,930,X,95,NaN,0,0,NaN,NaN,HENDRA,NaN
3,4,14128,CN800,0418882000,F,935,D,NaN,75,0,0,NaN,NaN,HENDRA,NaN
4,5,14129,JH2211,0412771876,F,945,X,NaN,95,0,0,NaN,NaN,HENDRA,NaN
5,6,14130,CE32DD,0416155783,F,1010,D,NaN,75,0,0,NaN,NaN,SHIVAM,NaN
6,7,14131,EVU18U,0499080777,F,1030,GV,NaN,NaN,0,0,300,NaN,SHIVAM,XD-50-176 TO 181
7,8,14132,EXD61Y,0407213067,F,1050,D,NaN,75,0,0,NaN,NaN,SHIVAM,NaN
8,9,14133,DTY90V,0439592230,M,1100,H,NaN,100,0,0,NaN,NaN,SHIVAM,NaN
9,10,14134,NXQ74D,0408518552,F,1100,X,75,NaN,0,0,NaN,NaN,SHIVAM,NaN


In [42]:
#give us summary of all df_{day}_part{idx} dataframes - number of rows, columns, and column names
for day in days:
    for idx in range(1, len(iloc_ranges)+1):
        name = f"df_{day}_part{idx}"
        if name not in globals():
            continue
        df_local = globals()[name]
        print(f"{name}: rows {len(df_local)} cols {len(df_local.columns)} columns: {list(df_local.columns)}")

df_mon_part1: rows 21 cols 15 columns: ['No', 'Docket No', 'Reg No', 'Mobile', 'Gender', 'TIME', 'Service', 'CASH', 'CARD', 'RSL', 'GV', 'GV CASH', 'GV CARD', 'O/Taker', ' Comment']
df_tue_part1: rows 16 cols 15 columns: ['No', 'Docket No', 'Reg No', 'Mobile', 'Gender', 'TIME', 'Service', 'CASH', 'CARD', 'RSL', 'GV', 'GV CASH', 'GV CARD', 'O/Taker', ' Comment']
df_wed_part1: rows 20 cols 15 columns: ['No', 'Docket No', 'Reg No', 'Mobile', 'Gender', 'TIME', 'Service', 'CASH', 'CARD', 'RSL', 'GV', 'GV CASH', 'GV CARD', 'O/Taker', ' Comment']
df_thu_part1: rows 21 cols 15 columns: ['No', 'Docket No', 'Reg No', 'Mobile', 'Gender', 'TIME', 'Service', 'CASH', 'CARD', 'RSL', 'GV', 'GV CASH', 'GV CARD', 'O/Taker', ' Comment']
df_fri_part1: rows 19 cols 15 columns: ['No', 'Docket No', 'Reg No', 'Mobile', 'Gender', 'TIME', 'Service', 'CASH', 'CARD', 'RSL', 'GV', 'GV CASH', 'GV CARD', 'O/Taker', ' Comment']
df_sat_part1: rows 25 cols 15 columns: ['No', 'Docket No', 'Reg No', 'Mobile', 'Gender', '

In [43]:
# join them all in the same dataframe with additional column for day and part
df_all = pd.DataFrame()
for day in days:
    for idx in range(1, len(iloc_ranges)+1):
        name = f"df_{day}_part{idx}"
        if name not in globals():
            continue
        df_local = globals()[name]
        df_local['Day'] = day.upper()
        df_local['Part'] = idx
        df_all = pd.concat([df_all, df_local], ignore_index=True)

In [44]:
df_all

,No,Docket No,Reg No,Mobile,Gender,TIME,Service,CASH,CARD,RSL,GV,GV CASH,GV CARD,O/Taker,Comment,Day,Part
0,1,14125,KEN838,0412038383,M,905,PCL,NaN,NaN,0,79,NaN,NaN,HENDRA,22012 (6/6),MON,1
1,2,14126,ENA66S,0416951959,F,915,PCL,NaN,NaN,0,79,NaN,475,HENDRA,23225 (1/6),MON,1
2,3,14127,EEY61G,0401995714,F,930,X,95,NaN,0,0,NaN,NaN,HENDRA,NaN,MON,1
3,4,14128,CN800,0418882000,F,935,D,NaN,75,0,0,NaN,NaN,HENDRA,NaN,MON,1
4,5,14129,JH2211,0412771876,F,945,X,NaN,95,0,0,NaN,NaN,HENDRA,NaN,MON,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,12,14262,DQR27A,0419487797,F,1345,X,NaN,95,0,0,NaN,NaN,HENDRA,NaN,SUN,1
134,13,14263,ERX40P,0405232683,M,1400,X,NaN,75,0,0,NaN,NaN,HENDRA,NaN,SUN,1
135,14,14264,EEN30S,0448544338,F,1415,D,NaN,75,0,0,NaN,NaN,HENDRA,NaN,SUN,1
136,15,14265,NBY11Q,0407197325,M,1420,X,NaN,95,0,0,NaN,NaN,HENDRA,NaN,SUN,1


In [45]:
# return this to excel
df_all.to_excel('../data/sample_cleaned.xlsx', index=False)